# 09. Diagnostics of the projected change and distributions of the hazard variables

This notebook produces the diagnostics of the CMIP6 projections and the wet-cell statistics 

## Imports

In [ ]:
import os, glob, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
warnings.filterwarnings("ignore")
%matplotlib inline

from matplotlib import font_manager
_fams = {f.name for f in font_manager.fontManager.ttflist}
FONT = next((f for f in ["Arial", "Helvetica", "Liberation Sans"] if f in _fams), "DejaVu Sans")
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": [FONT, "DejaVu Sans"],
    "font.size": 12, "axes.labelsize": 13, "axes.titlesize": 13,
    "xtick.labelsize": 11.5, "ytick.labelsize": 11.5, "legend.fontsize": 11,
    "axes.linewidth": 0.9, "xtick.major.width": 0.9, "ytick.major.width": 0.9,
    "xtick.direction": "out", "ytick.direction": "out",
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": True, "legend.fancybox": False, "legend.edgecolor": "0.25",
    "legend.framealpha": 1.0, "legend.borderpad": 0.5,
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.bbox": "tight", "savefig.pad_inches": 0.03})
print("font:", FONT)
print("ready")

## Configuration

In [ ]:
ROOT = Path("..")

# CMIP6 (13 GCMs, Mishra et al. 2020)
DATA_ROOT = str(ROOT / "data" / "cmip6")          # holds one folder per GCM
KHULNA_LAT, KHULNA_LON = 22.79, 89.57
HIST_WINDOW = (1985, 2014)
FUT_WINDOWS = {"2021–2050": (2021, 2050), "2051–2080": (2051, 2080)}
SSPS        = {"ssp245": "SSP2-4.5", "ssp585": "SSP5-8.5"}
TS_HIST_START = 1985      # first model year of the anomaly time series
TS_FUT_END    = 2100

# observed annual maximum series and change factors
OBS_FILE       = ROOT / "results" / "01_trend" / "AMS.csv"
CHANGE_FACTORS = ROOT / "results" / "03_cmip6" / "change_factors.csv"

# rasters and class breaks written by notebook 07
UFHI_DIR  = str(ROOT / "results" / "07_ufhi")
DEPTH_PAT = os.path.join(UFHI_DIR, "depth_masked_{rp}_{scen}.tif")
VEL_PAT   = os.path.join(UFHI_DIR, "velocity_masked_{rp}_{scen}.tif")
UFHI_PAT  = os.path.join(UFHI_DIR, "UFHI_{rp}_{scen}.tif")
RETURN_PERIODS = ["2yr", "10yr", "50yr", "100yr"]
SCENARIOS = ["Observed", "SSP245_2021_2050", "SSP245_2051_2080", "SSP585_2021_2050", "SSP585_2051_2080"]
HAZARD_PAIR     = ("Observed", "SSP585_2051_2080")   # left and right halves of the hazard violins
WET_THRESHOLD_M = 0.10     # depth defining a wet cell
N_SAMPLE        = 60000    # cells sampled per case for the density estimate (statistics use all cells)
UFHI_BREAKS     = pd.read_csv(os.path.join(UFHI_DIR, "jenks_breaks.csv"))["UFHI"].round(4).tolist()

# output
OUT_DIR   = str(ROOT / "results" / "09_diagnostics")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
DPI = 600
FIG_FORMATS = ("png", "tiff", "pdf")
COL = {"hist": "#9a9a9a", "ssp245": "#0072B2", "ssp585": "#D55E00", "gauge": "#009E73"}   # colour-blind-safe (Okabe and Ito)

# figure options
SHOW_POINTS  = False          # True overlays the individual annual values on the violins
SMOOTH_YEARS = 11             # centred running mean of the anomaly time series (years)
TRIM         = (0.005, 0.995) # display range of the pooled-anomaly violins
FIGB_LOG_AXIS      = True     # logarithmic rainfall axis in the per-GCM violins
SHOW_MEDIAN_CHANGE = False    # True prints the change in median beneath each per-GCM violin
WET_SHARE_ON = "D"            # hazard figure that carries the wet share under the x labels ("D", "V", "U" or None)
os.makedirs(CACHE_DIR, exist_ok=True)

## 1. Functions

The CMIP6 reader is that of notebook 03, written with `pandas` so that only the column of the selected grid cell is parsed. The extracted annual maxima are cached, so the raw files are read once.

Each violin is a kernel density estimate with a Gaussian kernel whose bandwidth follows the rule of Silverman (1986), $h = 0.9\,\min(\hat\sigma,\ \mathrm{IQR}/1.34)\,n^{-1/5}$. Densities are truncated at the data range, and each half is scaled to the same maximum width, so the plots compare shapes and not sample sizes (Hintze and Nelson 1998).

In [ ]:
# ---------- CMIP6 reading ----------
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0; p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin(math.radians(lat2 - lat1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def read_precipdata(path, tlat, tlon):
    with open(path) as f:
        first = f.readline().split(); second = f.readline().split()
    lons = np.array(first[3:], float); lats = np.array(second[3:], float)
    j = int(np.argmin([haversine_km(tlat, tlon, la, lo) for la, lo in zip(lats, lons)]))
    df = pd.read_csv(path, sep=r"\s+", header=None, skiprows=2, usecols=[0, 1, 2, 3 + j],
                     names=["y", "m", "d", "p"], engine="c")
    idx = pd.to_datetime(dict(year=df.y, month=df.m, day=df.d))
    return pd.Series(df.p.values, index=idx), float(haversine_km(tlat, tlon, lats[j], lons[j]))

def _raw_files(scenario):
    return sorted(glob.glob(os.path.join(DATA_ROOT, "*", scenario, "PrecipData*")))

def get_ams(scenario, y0, y1, verbose=True):
    # Per-model annual maximum daily rainfall, rows = years, columns = GCMs
    cache = os.path.join(CACHE_DIR, f"AMS_per_model_{scenario}_{y0}_{y1}.csv")
    if os.path.exists(cache):
        return pd.read_csv(cache, index_col=0)
    files = _raw_files(scenario)
    if files:
        out = {}
        for p in files:
            model = Path(p).parts[-3]
            s, dist = read_precipdata(p, KHULNA_LAT, KHULNA_LON)
            s = s[(s.index.year >= y0) & (s.index.year <= y1)]
            if len(s):
                out[model] = s.groupby(s.index.year).max()
        tab = pd.DataFrame(out); tab.index.name = "Year"
        tab.to_csv(cache)
        if verbose:
            print(f"  [{scenario} {y0}-{y1}] {tab.shape[1]} GCMs x {tab.shape[0]} years extracted (cell {dist:.1f} km from target)")
        return tab
    raise FileNotFoundError(f"No PrecipData files for scenario '{scenario}' under {DATA_ROOT}")

# ---------- GEV by L-moments (as in notebooks 02 and 03) ----------
def lmoments(x):
    x = np.sort(np.asarray(x, float)); n = len(x); j = np.arange(1, n + 1)
    b0 = x.mean(); b1 = np.sum((j - 1) / (n - 1) * x) / n
    b2 = np.sum((j - 1) * (j - 2) / ((n - 1) * (n - 2)) * x) / n
    l2 = 2 * b1 - b0
    return b0, l2, (6 * b2 - 6 * b1 + b0) / l2

def gev_q(x, Ts):
    l1, l2, t3 = lmoments(x)
    cc = 2.0 / (3.0 + t3) - math.log(2) / math.log(3)
    k = 7.8590 * cc + 2.9554 * cc ** 2
    k = k if abs(k) > 1e-6 else 1e-6
    gk = math.gamma(1 + k); scale = l2 * k / ((1 - 2 ** (-k)) * gk); loc = l1 - scale * (1 - gk) / k
    return {T: float(stats.genextreme.ppf(1 - 1.0 / T, c=k, loc=loc, scale=scale)) for T in Ts}

# ---------- trend statistics (as in notebook 01) ----------
def mann_kendall(x):
    x = np.asarray(x, float); n = len(x)
    s = sum(np.sum(np.sign(x[k + 1:] - x[k])) for k in range(n - 1))
    _, c = np.unique(x, return_counts=True); t = c[c > 1]
    var = (n * (n - 1) * (2 * n + 5) - np.sum(t * (t - 1) * (2 * t + 5))) / 18.0
    z = (s - 1) / math.sqrt(var) if s > 0 else (s + 1) / math.sqrt(var) if s < 0 else 0.0
    return s, z, 2 * (1 - stats.norm.cdf(abs(z)))

def sens_slope(t, x):
    t = np.asarray(t, float); x = np.asarray(x, float); sl = []
    for i in range(len(x) - 1):
        dt = t[i + 1:] - t[i]; dv = x[i + 1:] - x[i]; sl.extend(dv[dt != 0] / dt[dt != 0])
    b = float(np.median(sl)); return b, float(np.median(x - b * t))

# ---------- violin drawing ----------
def _kde(x, log=False, trim=None, n_grid=256):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if log: x = x[x > 0]
    t = np.log10(x) if log else x
    lo, hi = t.min(), t.max()
    if trim is not None: lo, hi = np.quantile(t, trim[0]), np.quantile(t, trim[1])
    sd = t.std(ddof=1); iqr = np.subtract(*np.percentile(t, [75, 25]))
    spread = min(sd, iqr / 1.34) if iqr > 0 else sd
    h = 0.9 * spread * len(t) ** (-0.2)
    if not np.isfinite(h) or h <= 0 or sd <= 0:
        grid = np.array([lo, hi if hi > lo else lo + 1e-9]); dens = np.ones(2)
    else:
        grid = np.linspace(lo, hi, n_grid); dens = gaussian_kde(t, bw_method=h / sd)(grid)
    return (10 ** grid if log else grid), dens

def half_violin(ax, x, pos, side, color, width=0.42, log=False, trim=None,
                points=None, alpha=0.9, lw_edge=0.8, quartiles=True, rng=None):
    # side: 'left', 'right' or 'both'. Returns the median of x.
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if log: x = x[x > 0]
    if len(x) < 3: return np.nan
    points = SHOW_POINTS if points is None else points
    y, d = _kde(x, log, trim); d = d / d.max() * width
    sides = ["left", "right"] if side == "both" else [side]
    for sd_ in sides:
        sgn = -1 if sd_ == "left" else 1
        ax.fill_betweenx(y, pos, pos + sgn * d, color=color, alpha=alpha, lw=0, zorder=2)
        ax.plot(pos + sgn * d, y, color="k", lw=lw_edge, zorder=3)
        for e in (0, -1):   # close the outline where the density is cut at the data range
            ax.plot([pos, pos + sgn * d[e]], [y[e], y[e]], color="k", lw=lw_edge, zorder=3)
        if quartiles:
            for q, ls, lw in [(25, (0, (2, 1.5)), 0.8), (50, "-", 1.5), (75, (0, (2, 1.5)), 0.8)]:
                v = np.percentile(x, q); w = np.interp(v, y, d)
                ax.plot([pos, pos + sgn * w], [v, v], color="k", ls=ls, lw=lw, zorder=4)
        if points:
            rng = rng or np.random.default_rng(0)
            jit = pos + sgn * rng.uniform(0.04, 0.55, len(x)) * np.interp(x, y, d)
            ax.scatter(jit, x, s=4, color="k", alpha=0.45, lw=0, zorder=5)
    ax.plot([pos, pos], [y.min(), y.max()], color="k", lw=0.6, zorder=3)
    return float(np.median(x))

def save(fig, name):
    for ext in FIG_FORMATS:
        kw = {"pil_kwargs": {"compression": "tiff_lzw"}} if ext == "tiff" else {}
        fig.savefig(os.path.join(OUT_DIR, f"{name}.{ext}"), dpi=DPI, **kw)
    print("saved", name, "(" + ", ".join(FIG_FORMATS) + ")")

def box_legend(ax_or_fig, handles, **kw):
    kw.setdefault("handlelength", 1.6); kw.setdefault("columnspacing", 1.2)
    return ax_or_fig.legend(handles=handles, **kw)

def log_ticks(ax, subs=(1, 2, 5)):
    from matplotlib.ticker import LogLocator, FuncFormatter, NullFormatter
    ax.set_yscale("log"); ax.yaxis.set_major_locator(LogLocator(base=10, subs=subs))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}")); ax.yaxis.set_minor_formatter(NullFormatter())

print("helpers ready")

## 2. Annual maxima of the 13 GCMs

The historical run is read from 1985 to 2014 and both SSP runs from 2015 to 2100. The analysis windows (1985 to 2014, 2021 to 2050 and 2051 to 2080) are sliced from these tables, so the time series and the violins use the same values as the delta-change calculation of notebook 03.

In [ ]:
Y0H, Y1H = min(TS_HIST_START, HIST_WINDOW[0]), HIST_WINDOW[1]
hist_full = get_ams("historical", Y0H, Y1H)
fut_full  = {s: get_ams(s, 2015, TS_FUT_END) for s in SSPS}

for t in [hist_full, *fut_full.values()]:
    t.index = t.index.astype(int)
MODELS = sorted(set(hist_full.columns) & set.intersection(*[set(v.columns) for v in fut_full.values()]))
hist_win = hist_full.loc[HIST_WINDOW[0]:HIST_WINDOW[1], MODELS]
fut_win = {(s, w): fut_full[s].loc[a:b, MODELS] for s in SSPS for w, (a, b) in FUT_WINDOWS.items()}

print(f"\n{len(MODELS)} common GCMs:", ", ".join(MODELS))
print(f"Historical window {HIST_WINDOW}: {hist_win.shape[0]} years; ensemble-mean AMS = {hist_win.mean(axis=1).mean():.1f} mm")
for (s, w), t in fut_win.items():
    print(f"{SSPS[s]} {w}: {t.shape[0]} years; ensemble-mean AMS = {t.mean(axis=1).mean():.1f} mm; "
          f"largest single-model value = {t.max().max():.0f} mm ({t.max().idxmax()})")

## 3. Check of the change factors

The change factors are recomputed from the tables above with the method of notebook 03 (GEV by L-moments fitted to the year-by-year ensemble-mean series) and compared with the values saved by that notebook.

In [ ]:
TS = [2, 5, 10, 25, 50, 100]
ref = pd.read_csv(CHANGE_FACTORS).set_index(["Case", "Return period (yr)"])["Change factor"]
ver = []
hq = gev_q(hist_win.mean(axis=1).values, TS)
for (s, w), t in fut_win.items():
    fq = gev_q(t.mean(axis=1).values, TS)
    case = f"{s}_{FUT_WINDOWS[w][0]}_{FUT_WINDOWS[w][1]}"
    for T in TS:
        cf = fq[T] / hq[T]; r = ref.get((case, T))
        ver.append({"Scenario": SSPS[s], "Window": w, "T (yr)": T, "Hist (mm)": round(hq[T], 1),
                    "Future (mm)": round(fq[T], 1), "CF (recomputed)": round(cf, 3),
                    "CF (notebook 03)": r, "Match": "" if r is None else ("yes" if abs(cf - r) < 0.005 else "no")})
ver = pd.DataFrame(ver); ver.to_csv(os.path.join(OUT_DIR, "change_factor_check.csv"), index=False)
display(ver.pivot_table(index="T (yr)", columns=["Scenario", "Window"], values="CF (recomputed)"))
bad = ver[ver.Match == "no"]
print("All change factors of notebook 03 reproduced." if bad.empty else f"{len(bad)} value(s) differ:\n{bad}")

q2 = {w: ver.query("Scenario=='SSP2-4.5' and Window==@w and `T (yr)`==2")["Future (mm)"].item() for w in FUT_WINDOWS}
print("\n2-year GEV quantile of the SSP2-4.5 ensemble-mean series by window (mm):", q2)

## 4. Sensitivity of the 100-year change factor to the construction of the ensemble

Averaging the annual maxima of 13 models before fitting the GEV gives a series with far less variance than any single model, which makes the change factor in the tail sensitive to one or two extreme model-years. Four estimates of the 100-year change factor are compared:

1. **Adopted method:** GEV fitted to the year-by-year ensemble-mean series.
2. **Per-model factors:** GEV fitted to the 30 annual maxima of each model; median and interquartile range of the 13 factors.
3. **Pooled sample:** GEV fitted to all 13 × 30 annual maxima of each period.
4. **Leave-one-out:** the adopted method repeated 13 times with one model omitted.

In [ ]:
T = 100; rob = []
for (s, w), t in fut_win.items():
    m1 = gev_q(t.mean(axis=1), [T])[T] / gev_q(hist_win.mean(axis=1), [T])[T]
    per = []
    for m in MODELS:
        try: per.append(gev_q(t[m].dropna(), [T])[T] / gev_q(hist_win[m].dropna(), [T])[T])
        except Exception: pass
    per = np.array(per)
    m3 = gev_q(t.values.ravel(), [T])[T] / gev_q(hist_win.values.ravel(), [T])[T]
    loo = {m: gev_q(t.drop(columns=m).mean(axis=1), [T])[T] / gev_q(hist_win.drop(columns=m).mean(axis=1), [T])[T]
           for m in MODELS}
    kmin, kmax = min(loo, key=loo.get), max(loo, key=loo.get)
    rob.append({"Scenario": SSPS[s], "Window": w,
                "CF adopted (ensemble mean)": round(m1, 3),
                "CF per-model median": round(np.median(per), 3),
                "CF per-model IQR": f"{np.percentile(per,25):.2f}–{np.percentile(per,75):.2f}",
                "CF pooled": round(m3, 3),
                "Leave-one-out minimum (model omitted)": f"{loo[kmin]:.3f} ({kmin})",
                "Leave-one-out maximum (model omitted)": f"{loo[kmax]:.3f} ({kmax})"})
rob = pd.DataFrame(rob); rob.to_csv(os.path.join(OUT_DIR, "cf100_sensitivity.csv"), index=False)
display(rob)

## 5. Observed and projected annual maximum rainfall

Four panels are written as separate files:

- the observed annual maximum daily rainfall at Khulna with the Sen slope;
- the gauge annual maxima against the pooled annual maxima of the 13 GCMs in the historical period, on the same rainfall axis;
- the annual maxima of each GCM as a percentage change from its own 1985 to 2014 mean, $a_{g,y} = 100\,(P_{g,y} - \overline{P}_{g,\mathrm{hist}})/\overline{P}_{g,\mathrm{hist}}$, shown as the 13-model median and the inter-model interquartile range, both smoothed with an 11-year centred running mean;
- the anomalies pooled over each 30-year window as split violins, drawn between the 0.5th and 99.5th percentiles.

In [ ]:
# ---------- data ----------
obs = pd.read_csv(OBS_FILE); obs.columns = [c.strip() for c in obs.columns]
dcol = next((c for c in obs.columns if c.lower() == "date"), None)
vcol = next(c for c in obs.columns[::-1] if c.lower() not in ("year", "date") and pd.api.types.is_numeric_dtype(obs[c]))
if dcol:
    obs[dcol] = pd.to_datetime(obs[dcol], errors="coerce"); obs = obs.dropna(subset=[dcol])
    ams_obs = obs.groupby(obs[dcol].dt.year)[vcol].max()
else:
    ycol = "Year" if "Year" in obs.columns else obs.columns[0]
    ams_obs = obs.set_index(ycol)[vcol]
ams_obs = ams_obs.dropna().astype(float).sort_index(); ams_obs.index = ams_obs.index.astype(int)
low = ams_obs[ams_obs < 30]
if len(low): print(f"Note: annual maxima below 30 mm in {list(low.index)}")
S, Z, P = mann_kendall(ams_obs.values); sen_b, sen_a = sens_slope(ams_obs.index, ams_obs.values)
print(f"Gauge AMS {ams_obs.index.min()}-{ams_obs.index.max()} (n={len(ams_obs)}): MK Z={Z:.3f}, p={P:.4f}; Sen={sen_b:.4f} mm/yr")

base = hist_win.mean()
anom_h = 100 * (hist_full[MODELS] / base - 1)
anom_f = {s: 100 * (fut_full[s][MODELS] / base - 1) for s in SSPS}
def pooled(tab, a, b): return tab.loc[a:b].values.ravel()
gcm_h_abs = hist_win.values.ravel()

YLAB_AMS  = "Annual maximum daily rainfall (mm)"
YLAB_ANOM = "Change in annual maximum daily\nrainfall relative to 1985–2014 (%)"
win_lab = lambda a, b: f"{a}–{b}"
YL_A12 = (0, max(ams_obs.max(), gcm_h_abs.max()) * 1.06)

In [ ]:
# ---------- observed gauge series ----------
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.bar(ams_obs.index, ams_obs.values, width=0.75, color=COL["gauge"], edgecolor="none")
yy = np.arange(ams_obs.index.min(), ams_obs.index.max() + 1)
ax.plot(yy, sen_a + sen_b * yy, color="k", lw=1.8)
ax.set_xlim(ams_obs.index.min() - 1, ams_obs.index.max() + 1); ax.set_ylim(*YL_A12)
ax.set_xlabel("Year"); ax.set_ylabel(YLAB_AMS)
box_legend(ax, [Patch(color=COL["gauge"], label="Observed annual maximum"),
                Line2D([], [], color="k", lw=1.8,
                       label=f"Sen's slope: {sen_b:+.2f} mm yr$^{{-1}}$ (Mann–Kendall p = {P:.2f})")],
           loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2)
save(fig, "fig_gauge_ams_timeseries"); plt.show()

In [ ]:
# ---------- gauge against GCM grid cell, historical ----------
fig, ax = plt.subplots(figsize=(4.2, 4.4))
mg = half_violin(ax, ams_obs.values, 0, "left", COL["gauge"])
mh = half_violin(ax, gcm_h_abs, 0, "right", COL["hist"])
ax.set_xticks([-0.22, 0.22])
ax.set_xticklabels([f"Rain gauge\n{ams_obs.index.min()}–{ams_obs.index.max()}", f"CMIP6 GCMs\n{win_lab(*HIST_WINDOW)}"])
ax.set_xlim(-0.6, 0.6); ax.set_ylim(*YL_A12)
ax.set_ylabel(YLAB_AMS)
save(fig, "fig_gauge_vs_gcm_violin"); plt.show()
print(f"Median gauge AMS = {mg:.0f} mm (n = {len(ams_obs)}); median GCM AMS = {mh:.0f} mm (n = {gcm_h_abs.size})")

In [ ]:
# ---------- anomaly time series ----------
def smooth_q(tab):
    q = tab.quantile([0.25, 0.5, 0.75], axis=1).T
    return q.rolling(SMOOTH_YEARS, center=True, min_periods=max(1, SMOOTH_YEARS // 2 + 1)).mean() if SMOOTH_YEARS > 1 else q
Q = {"hist": smooth_q(anom_h), **{s: smooth_q(anom_f[s]) for s in SSPS}}
lo = min(q[0.25].min() for q in Q.values()); hi = max(q[0.75].max() for q in Q.values())
span = hi - lo; YL_A3 = (lo - 0.08 * span, hi + 0.22 * span)

fig, ax = plt.subplots(figsize=(9, 4.4))
XL = (min(TS_HIST_START, HIST_WINDOW[0]) - 1, TS_FUT_END + 1)
for (a, b) in [HIST_WINDOW, *FUT_WINDOWS.values()]:
    for edge in (a - 0.5, b + 0.5):
        ax.axvline(edge, color="0.55", lw=0.8, ls=(0, (3, 3)), zorder=0)
    ax.text((a + b) / 2, 0.97, win_lab(a, b), transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=11, color="0.25")
handles = []
for key, lab in [("hist", "Historical"), *[(s, SSPS[s]) for s in SSPS]]:
    q = Q[key]
    ax.fill_between(q.index, q[0.25], q[0.75], color=COL[key], alpha=0.22, lw=0, zorder=1)
    ax.plot(q.index, q[0.5], color=COL[key], lw=2.2, zorder=3)
    handles.append(Line2D([], [], color=COL[key], lw=2.2, label=lab))
handles.append(Patch(color="0.6", alpha=0.35, label="Inter-model IQR"))
ax.axhline(0, color="k", lw=0.8, zorder=2)
ax.set_xlim(*XL); ax.set_ylim(*YL_A3)
ax.set_xlabel("Year"); ax.set_ylabel(YLAB_ANOM)
box_legend(ax, handles, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=4)
save(fig, "fig_cmip6_anomaly_timeseries"); plt.show()

In [ ]:
# ---------- pooled anomalies by window ----------
def qrange(v, tr): return np.quantile(v[np.isfinite(v)], tr[0]), np.quantile(v[np.isfinite(v)], tr[1])
sets = [pooled(anom_h, *HIST_WINDOW)] + [pooled(anom_f[s], a, b) for s in SSPS for (a, b) in FUT_WINDOWS.values()]
rng_v = np.array([qrange(v, TRIM) for v in sets]); span = rng_v[:, 1].max() - rng_v[:, 0].min()
YL_A4 = (rng_v[:, 0].min() - 0.04 * span, rng_v[:, 1].max() + 0.04 * span)
n_out = sum(int(((v < qrange(v, TRIM)[0]) | (v > qrange(v, TRIM)[1])).sum()) for v in sets)
print(f"Violins are drawn between the {100*TRIM[0]:.1f}th and {100*TRIM[1]:.1f}th percentiles "
      f"({n_out} of {sum(v.size for v in sets)} values lie outside and are included in all statistics).")

fig, ax = plt.subplots(figsize=(5.6, 4.4))
half_violin(ax, sets[0], 0, "both", COL["hist"], trim=TRIM)
labels = [win_lab(*HIST_WINDOW)]
for i, (w, (a, b)) in enumerate(FUT_WINDOWS.items(), start=1):
    half_violin(ax, pooled(anom_f["ssp245"], a, b), i, "left", COL["ssp245"], trim=TRIM)
    half_violin(ax, pooled(anom_f["ssp585"], a, b), i, "right", COL["ssp585"], trim=TRIM)
    labels.append(win_lab(a, b))
ax.axhline(0, color="k", lw=0.8, zorder=1)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
ax.set_xlim(-0.6, len(labels) - 0.4); ax.set_ylim(*YL_A4)
ax.set_xlabel("Period"); ax.set_ylabel(YLAB_ANOM)
box_legend(ax, [Patch(color=COL["hist"], label="Historical"),
                Patch(color=COL["ssp245"], label="SSP2-4.5 (left)"),
                Patch(color=COL["ssp585"], label="SSP5-8.5 (right)")],
           loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3, fontsize=10.5,
           handlelength=1.0, handletextpad=0.4, columnspacing=0.8)
save(fig, "fig_cmip6_anomaly_violins"); plt.show()

rows = [{"Window": win_lab(*HIST_WINDOW), "Scenario": "Historical",
         "Median anomaly (%)": round(np.median(sets[0]), 1), "P90 anomaly (%)": round(np.percentile(sets[0], 90), 1)}]
for s in SSPS:
    for w, (a, b) in FUT_WINDOWS.items():
        v = pooled(anom_f[s], a, b)
        rows.append({"Window": w, "Scenario": SSPS[s], "Median anomaly (%)": round(np.median(v), 1),
                     "P90 anomaly (%)": round(np.percentile(v, 90), 1)})
figA_tab = pd.DataFrame(rows); figA_tab.to_csv(os.path.join(OUT_DIR, "anomaly_summary_by_window.csv"), index=False); display(figA_tab)

## 6. Per-GCM distributions

Each violin pairs the 1985 to 2014 annual maxima of one model (grey, left half) with its future annual maxima (coloured, right half). The first position is the year-by-year ensemble-mean series from which the change factors are derived. The rainfall axis is logarithmic. The table gives the change in the median annual maximum of each model and the number of models with an increase.

In [ ]:
order = ["Ensemble mean"] + MODELS
allB = np.concatenate([hist_win.values.ravel()] + [fut_win[k].values.ravel() for k in fut_win])
YL_B = (np.nanmin(allB) * 0.88, np.nanmax(allB) * 1.12) if FIGB_LOG_AXIS else (0, np.nanpercentile(allB, 99.8) * 1.05)

def figB_one(s, w):
    fig, ax = plt.subplots(figsize=(10, 5.0))
    fw = fut_win[(s, w)]
    for i, m in enumerate(order):
        h = hist_win.mean(axis=1) if i == 0 else hist_win[m]
        f = fw.mean(axis=1) if i == 0 else fw[m]
        mh = half_violin(ax, h, i, "left", COL["hist"], log=FIGB_LOG_AXIS)
        mf = half_violin(ax, f, i, "right", COL[s], log=FIGB_LOG_AXIS)
        if SHOW_MEDIAN_CHANGE:
            dv = 100 * (mf / mh - 1); dv = 0.0 if abs(dv) < 0.5 else dv
            ax.text(i, 0.02, f"{dv:+.0f}%", transform=ax.get_xaxis_transform(), ha="center", fontsize=9.5, color="0.3")
    if FIGB_LOG_AXIS: log_ticks(ax, subs=(1, 2, 3, 5))
    ax.set_ylim(*YL_B)
    ax.axvline(0.5, color="0.55", lw=0.9, ls=(0, (3, 3)), zorder=0)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=45, ha="right", rotation_mode="anchor")
    ax.set_xlim(-0.6, len(order) - 0.4)
    ax.set_xlabel("Global climate model"); ax.set_ylabel(YLAB_AMS)
    box_legend(ax, [Patch(color=COL["hist"], label=f"Historical {win_lab(*HIST_WINDOW)} (left)"),
                    Patch(color=COL[s], label=f"{SSPS[s]}, {w} (right)")],
               loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2)
    return fig

for s in SSPS:
    for w in FUT_WINDOWS:
        fig = figB_one(s, w); save(fig, f"fig_per_gcm_violins_{s}_{w.replace('–', '_')}"); plt.show()

tabB = pd.DataFrame({f"{SSPS[s]} {w}": (fut_win[(s, w)].median() / hist_win.median() - 1).mul(100).round(1)
                     for s in SSPS for w in FUT_WINDOWS})
tabB.loc["Ensemble mean"] = [round(100 * (fut_win[(s, w)].mean(axis=1).median() / hist_win.mean(axis=1).median() - 1), 1)
                             for s in SSPS for w in FUT_WINDOWS]
tabB.loc["Models with increase (of 13)"] = [(fut_win[(s, w)].median() > hist_win.median()).sum() for s in SSPS for w in FUT_WINDOWS]
tabB.to_csv(os.path.join(OUT_DIR, "median_change_by_model.csv")); display(tabB)

## 7. Depth, velocity and UFHI over wet cells

Cells with a maximum depth of at least 0.10 m are defined as wet. For each of the 20 cases the notebook computes the wet share of the modelled domain, the median, 90th and 99th percentiles of depth, and the median and 90th percentile of velocity and of the UFHI over all wet cells. The modelled domain is taken as the union of the cells that hold a simulated depth in at least one of the 20 masked depth rasters. The violins compare the observed climate (left halves) with SSP5-8.5, 2051 to 2080 (right halves); their densities are estimated from a random sample of 60,000 wet cells per case. Depth and UFHI are drawn on logarithmic axes up to their 99.9th percentiles and velocity up to its 99.5th percentile; the dashed lines in the UFHI figure are the class breaks of notebook 07.

In [ ]:
import rasterio

def read_arr(p):
    with rasterio.open(p) as src:
        return src.read(1, masked=True).astype("float64").filled(np.nan)

domain = None; missing = []
for rp in RETURN_PERIODS:
    for s in SCENARIOS:
        p = DEPTH_PAT.format(rp=rp, scen=s)
        if not os.path.exists(p): missing.append(p); continue
        f = np.isfinite(read_arr(p)); domain = f if domain is None else (domain | f)
if missing: print(f"{len(missing)} depth raster(s) missing, e.g.\n  {missing[0]}")
NDOM = int(domain.sum()); print("Domain cells:", NDOM)

rng = np.random.default_rng(42); samp = {}; rows = []
for rp in RETURN_PERIODS:
    for s in SCENARIOS:
        pD = DEPTH_PAT.format(rp=rp, scen=s)
        if not os.path.exists(pD): continue
        D = read_arr(pD); V = read_arr(VEL_PAT.format(rp=rp, scen=s)); U = read_arr(UFHI_PAT.format(rp=rp, scen=s))
        wet = np.isfinite(D) & (D >= WET_THRESHOLD_M)
        d, v, u = D[wet], V[wet], U[wet]
        rows.append({"RP": rp, "Scenario": s, "Wet cells (%)": round(100 * wet.sum() / NDOM, 2),
                     "Depth median (m)": round(np.nanmedian(d), 3), "Depth P90 (m)": round(np.nanpercentile(d, 90), 3),
                     "Depth P99 (m)": round(np.nanpercentile(d, 99), 3),
                     "Velocity median (m/s)": round(np.nanmedian(v), 3), "Velocity P90 (m/s)": round(np.nanpercentile(v, 90), 3),
                     "UFHI median": round(np.nanmedian(u), 4), "UFHI P90": round(np.nanpercentile(u, 90), 4)})
        if s in HAZARD_PAIR:
            k = rng.choice(d.size, size=min(N_SAMPLE, d.size), replace=False)
            samp[(rp, s)] = dict(D=d[k], V=v[k], U=u[k], wet=100 * wet.sum() / NDOM)
        del D, V, U
haz = pd.DataFrame(rows); haz.to_csv(os.path.join(OUT_DIR, "wet_cell_statistics.csv"), index=False)
display(haz)

In [ ]:
from matplotlib.transforms import blended_transform_factory
left, right = HAZARD_PAIR
c_right = COL["ssp585"] if "585" in right else COL["ssp245"]
nice = lambda s: s.replace("SSP245_", "SSP2-4.5, ").replace("SSP585_", "SSP5-8.5, ").replace("_", "–")
VARS = [("D", "Depth (m)", True, (0.0, 0.999), "fig_wet_cell_depth"),
        ("V", "Velocity (m s$^{-1}$)", False, (0.0, 0.995), "fig_wet_cell_velocity"),
        ("U", "UFHI (–)", True, (0.0, 0.999), "fig_wet_cell_ufhi")]
rps = [rp for rp in RETURN_PERIODS if (rp, left) in samp and (rp, right) in samp]

for key, lab, lg, tr, fname in VARS:
    fig, ax = plt.subplots(figsize=(7.6, 4.6))
    for i, rp in enumerate(rps):
        half_violin(ax, samp[(rp, left)][key], i, "left", COL["hist"], width=0.44, log=lg, trim=tr)
        half_violin(ax, samp[(rp, right)][key], i, "right", c_right, width=0.44, log=lg, trim=tr)
    if lg: log_ticks(ax)
    if key == "V": ax.set_ylim(bottom=0)
    if key == "U":
        lims = ax.get_ylim(); tr_ = blended_transform_factory(ax.transAxes, ax.transData)
        for b in UFHI_BREAKS: ax.axhline(b, color="0.4", lw=0.8, ls=(0, (4, 3)), zorder=1)
        edges = [lims[0]] + UFHI_BREAKS + [lims[1]]
        for k, nm in enumerate(["Very Low", "Low", "Medium", "High", "Very High"]):
            mid = 10 ** ((np.log10(max(edges[k], 1e-6)) + np.log10(edges[k + 1])) / 2)
            if lims[0] < mid < lims[1]:
                ax.text(1.01, mid, nm, transform=tr_, fontsize=10.5, color="0.25", va="center", ha="left")
        ax.set_ylim(lims)
    ticks = [rp.replace("yr", "-year") for rp in rps]
    if WET_SHARE_ON == key:
        ticks = [f"{t}\n{samp[(rp,left)]['wet']:.1f}% → {samp[(rp,right)]['wet']:.1f}%" for t, rp in zip(ticks, rps)]
    ax.set_xticks(range(len(rps))); ax.set_xticklabels(ticks); ax.set_xlim(-0.6, len(rps) - 0.4)
    ax.set_xlabel("Return period\n(wet share of study area: observed → future)" if WET_SHARE_ON == key else "Return period")
    ax.set_ylabel(lab)
    box_legend(ax, [Patch(color=COL["hist"], label=f"{nice(left)} (left)"), Patch(color=c_right, label=f"{nice(right)} (right)")],
               loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2)
    save(fig, fname); plt.show()